# Fine-tune the F1 race engineer model

Teaches Qwen 2.5 1.5B to talk like a real F1 race engineer, using the real radio dataset you built with `tools/openf1_radio.py`. At the end you get one file to load into Ollama as `f1-engineer`.

**Before you start**
1. *Runtime > Change runtime type > T4 GPU* (free tier is fine).
2. Copy `data/openf1/examples_train.jsonl` and `data/openf1/examples_test.jsonl` from your PC to Google Drive, in a folder called `f1-engineer/openf1`.

Then *Runtime > Run all*. Training takes about 15-40 minutes on a T4.

The code still decides the call (strategy, gaps, pit window); this model only learns how to *say* it. Don't share the trained model or the transcripts publicly: the radio audio belongs to F1.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/f1-engineer/openf1'
OUT = '/content/drive/MyDrive/f1-engineer/model'
EPOCHS = 2
import os

for name in ('examples_train.jsonl', 'examples_test.jsonl'):
    path = f'{DATA}/{name}'
    assert os.path.exists(path), f'Copy {name} from data/openf1 on your PC to {DATA} first'
print('Found the dataset in', DATA)

## 1. Turn the radio lines into chats
Same system prompt and data layout the engineer sends during a race, with the real engineer's line as the answer. Lines quoting numbers the data doesn't have are left out.

In [ ]:
!rm -rf /content/engineer && git clone -q --depth 1 https://github.com/Shryder25/F1-21_Race_Engineer /content/engineer
!python /content/engineer/tools/training/make_sft.py --data "$DATA"

## 2. Install Unsloth and load the base model

In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel

MAX_LEN = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen2.5-1.5B-Instruct', max_seq_length=MAX_LEN, load_in_4bit=True
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

## 3. Train

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

data = load_dataset('json', data_files={'train': f'{DATA}/sft_train.jsonl', 'test': f'{DATA}/sft_test.jsonl'})
data = data.map(
    lambda b: {'text': [tokenizer.apply_chat_template(m, tokenize=False) for m in b['messages']]},
    batched=True,
)
print(data)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data['train'],
    eval_dataset=data['test'],
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        eval_strategy='epoch',
        optim='adamw_8bit',
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        output_dir='/content/outputs',
        report_to='none',
        seed=3407,
    ),
)
# Learn only the engineer's answer, not the prompt.
trainer = train_on_responses_only(
    trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n'
)
trainer.train()

## 4. Check it on the held-out races
Compares the base model and the trained one on races it never saw: does it return the JSON the engineer needs, how long are its lines, and what does it actually say.

In [ ]:
import json
import re

import pandas as pd

FastLanguageModel.for_inference(model)
test = [json.loads(line) for line in open(f'{DATA}/sft_test.jsonl', encoding='utf-8')][:60]


def reply(messages):
    prompt = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    ids = tokenizer(prompt, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=120, temperature=0.6, do_sample=True)
    return tokenizer.decode(out[0][ids['input_ids'].shape[1] :], skip_special_tokens=True).strip()


def voice(text):
    try:
        return json.loads(re.search(r'\{.*\}', text, re.S).group(0)).get('voice') or None
    except (AttributeError, ValueError):
        return None


rows = []
for ex in test:
    with model.disable_adapter():
        base = reply(ex['messages'])
    tuned = reply(ex['messages'])
    rows.append(
        {
            'question': ex['messages'][1]['content'].split('\n')[0],
            'real engineer': json.loads(ex['messages'][2]['content'])['voice'],
            'base model': voice(base) or base,
            'trained': voice(tuned) or tuned,
            'base_json': voice(base) is not None,
            'trained_json': voice(tuned) is not None,
        }
    )
df = pd.DataFrame(rows)


def words(column):
    return df[column].str.split().str.len()


print(f"Valid JSON: base {df.base_json.mean():.0%}, trained {df.trained_json.mean():.0%}")
print(
    f"Average words: real {words('real engineer').mean():.1f}, base {words('base model').mean():.1f}, trained {words('trained').mean():.1f}"
)
print(
    f"Under 25 words: base {(words('base model') <= 25).mean():.0%}, trained {(words('trained') <= 25).mean():.0%}"
)
pd.set_option('display.max_colwidth', 120)
df[['question', 'real engineer', 'base model', 'trained']].head(20)

## 5. Save it for Ollama
Writes a ~1 GB `.gguf` file and a `Modelfile` to `f1-engineer/model` on your Drive.

In [ ]:
import glob
import os

model.save_pretrained_gguf(f'{OUT}/gguf', tokenizer, quantization_method='q4_k_m')
found = glob.glob(f'{OUT}/gguf/*.gguf') + glob.glob(f'{OUT}/gguf*/*.gguf')
q4 = [f for f in found if 'q4_k_m' in os.path.basename(f).lower()]
gguf = (q4 or sorted(found, key=os.path.getsize))[0]  # the 4-bit file, not the big f16 one
os.replace(gguf, f'{OUT}/f1-engineer.gguf')
template = '''{{- range .Messages }}<|im_start|>{{ .Role }}
{{ .Content }}<|im_end|>
{{ end }}<|im_start|>assistant
'''
with open(f'{OUT}/Modelfile', 'w') as f:
    f.write('FROM ./f1-engineer.gguf\n')
    f.write(f'TEMPLATE """{template}"""\n')
    f.write('PARAMETER stop "<|im_end|>"\nPARAMETER temperature 0.6\nPARAMETER num_ctx 2048\n')
print('Saved', f'{OUT}/f1-engineer.gguf', round(os.path.getsize(f'{OUT}/f1-engineer.gguf') / 1e9, 2), 'GB')

## 6. Use it on your PC

1. Download `f1-engineer.gguf` and `Modelfile` from `f1-engineer/model` on Google Drive into one folder.
2. In that folder run: `ollama create f1-engineer -f Modelfile`
3. Test it with the game closed: `python main.py --ai-model f1-engineer --ai-test`
4. Race with it: `python main.py --ai-model f1-engineer`

If it's worse than plain `qwen2.5:1.5b`, go back to that model. Nothing else changes.